# Módulo 4 · Formatos: CSV, Parquet y Delta

⏱ **15 min** · Slides: 24–26 (por filas vs por columnas, comparativo, Delta = Parquet + log)

**Objetivo:** guardar *los mismos datos* en tres formatos y medir las diferencias.

| Paso | Qué demuestras |
|---|---|
| 1 | Preparar un DataFrame plano |
| 2 | Escribir en CSV, Parquet y Delta |
| 3 | **Tamaño**: ¿cuál pesa menos? |
| 4 | **Esquema**: ¿quién recuerda los tipos? |
| 5 | **Columnar**: ¿qué columnas se leen? |
| 6 | **Delta = Parquet + `_delta_log`** |
| 7 | **ACID**: una escritura fallida no daña la tabla |
| 8 | **Time travel**: volver a una versión anterior |

**Requiere:** módulo 2.

In [ ]:
%run ./00_Setup

## Paso 1 · Preparar un DataFrame plano

CSV no soporta estructuras anidadas ni listas, así que aplanamos el JSON a columnas simples.

In [ ]:
plano = read_raw().select(
    F.col("id").alias("episode_id"),
    F.col("name").alias("episode_name"),
    "season",
    F.col("number").alias("episode_number"),
    F.col("runtime").alias("runtime_min"),
    F.col("rating.average").alias("episode_rating"),
    "airdate",
    F.col("_embedded.show.id").alias("show_id"),
    F.col("_embedded.show.name").alias("show_name"),
    F.col("_embedded.show.language").alias("show_language"),
    F.concat_ws(",", F.col("_embedded.show.genres")).alias("show_genres"),   # lista → texto
    F.col("_embedded.show.summary").alias("show_summary"),
)

print("Filas:", plano.count())
display(plano.limit(5))

## Paso 2 · Escribir en los tres formatos

In [ ]:
csv_path     = f"{FORMATOS_PATH}/csv"
parquet_path = f"{FORMATOS_PATH}/parquet"
delta_path   = f"{FORMATOS_PATH}/delta"

plano.write.mode("overwrite").option("header", True).csv(csv_path)
plano.write.mode("overwrite").parquet(parquet_path)
plano.write.mode("overwrite").format("delta").save(delta_path)

print("✔ Tres copias escritas en", FORMATOS_PATH)

## Paso 3 · Tamaño en disco

In [ ]:
tamanos = [("CSV", size_mb(csv_path)), ("Parquet", size_mb(parquet_path)), ("Delta", size_mb(delta_path))]
display(spark.createDataFrame(tamanos, "formato STRING, mb DOUBLE"))

👀 **Qué observar:** los mismos datos pesan mucho menos en Parquet (binario, comprimido).
Delta pesa casi igual que Parquet: por dentro **son archivos Parquet** (más un pequeño log).
Pista: usa el selector de gráfico de barras sobre la tabla.

## Paso 4 · Esquema: ¿quién recuerda los tipos?

In [ ]:
print("CSV sin esquema → todo texto:")
spark.read.option("header", True).csv(csv_path).select("episode_id", "runtime_min", "airdate").printSchema()

In [ ]:
print("Parquet → tipos incluidos en el archivo:")
spark.read.parquet(parquet_path).select("episode_id", "runtime_min", "airdate").printSchema()

## Paso 5 · Columnar: leer solo lo necesario

Consulta: `AVG(runtime_min)`. Solo se necesita **1 de 12 columnas**. Busca `ReadSchema` en cada plan.

In [ ]:
print("===== CSV =====")
spark.read.option("header", True).csv(csv_path).agg(F.avg("runtime_min")).explain()

In [ ]:
print("===== PARQUET =====")
spark.read.parquet(parquet_path).agg(F.avg("runtime_min")).explain()

👀 **Qué observar:** en Parquet, `ReadSchema` lista **solo `runtime_min`** y con su tipo real (`int`).
En CSV también se podan columnas, pero cada línea completa debe leerse y analizarse como texto,
y `runtime_min` llega como `string`. Con muchos GB, esa diferencia es el 100 % vs ~10 % de lectura del slide 24.

## Paso 6 · Delta = Parquet + un log de transacciones

In [ ]:
display(dbutils.fs.ls(delta_path))

In [ ]:
display(dbutils.fs.ls(f"{delta_path}/_delta_log"))

In [ ]:
# La primera transacción: el JSON del log lista los archivos Parquet que "pertenecen" a la tabla
log0 = spark.read.json(f"{delta_path}/_delta_log/00000000000000000000.json")
display(log0.where("add IS NOT NULL").select("add.path", "add.size"))

👀 **Qué observar:** la carpeta tiene archivos `.parquet` y una carpeta `_delta_log` con JSON.
Una tabla Delta **es lo que dice el log**: un archivo Parquet que no aparece en el log, no existe para la tabla.
Ese log es lo que da transacciones y versiones.

## Paso 7 · ACID: todo o nada

Simulamos un job que **falla a mitad de una escritura** (`overwrite`) sobre la tabla Delta.

In [ ]:
from delta.tables import DeltaTable

tabla = DeltaTable.forPath(spark, delta_path)

def current_state():
    filas = spark.read.format("delta").load(delta_path).count()
    version = tabla.history().agg(F.max("version")).first()[0]
    return filas, version

filas_antes, version_antes = current_state()
print(f"ANTES   → filas: {filas_antes} | versión: {version_antes}")

In [ ]:
# raise_error() hace fallar la escritura cuando llega a ciertas filas
roto = plano.withColumn(
    "runtime_min",
    F.when(F.col("episode_id") % 20 == 0, F.expr("raise_error('Fallo simulado a mitad de la escritura')"))
     .otherwise(F.col("runtime_min")))

try:
    roto.write.mode("overwrite").format("delta").save(delta_path)
except Exception as e:
    print("❌ La escritura falló:", str(e).splitlines()[0][:150])

In [ ]:
filas_despues, version_despues = current_state()
print(f"DESPUÉS → filas: {filas_despues} | versión: {version_despues}")
print("¿Tabla intacta? ", (filas_antes, version_antes) == (filas_despues, version_despues))

👀 **Qué observar:** a pesar del error, la tabla quedó **exactamente igual**: mismas filas, misma versión.
El commit nunca se registró en el log, así que nadie ve una escritura a medias.

## Paso 8 · Time travel: volver atrás

Cada cambio exitoso crea una versión nueva. Borremos filas "por error" y recuperémoslas.

In [ ]:
tabla.delete("episode_rating IS NULL")            # "error": borra todo episodio sin calificación
print("Filas tras el DELETE:", spark.read.format("delta").load(delta_path).count())

In [ ]:
display(tabla.history().select("version", "operation", "operationMetrics"))

In [ ]:
# Leer la versión 0 (antes del DELETE)
v0 = spark.read.format("delta").option("versionAsOf", 0).load(delta_path)
print("Filas en la versión 0:", v0.count())

In [ ]:
tabla.restoreToVersion(0)
print("Filas tras RESTORE a la versión 0:", spark.read.format("delta").load(delta_path).count())

👀 **Qué observar:** `DELETE` creó la versión 1; `RESTORE` creó la 2 y devolvió los datos.
Con Parquet o CSV esto no es posible: los archivos son inmutables y no hay historial.

## Resumen del módulo

| | CSV | Parquet | Delta |
|---|---|---|---|
| Tamaño | Mayor | Menor | Menor |
| Esquema | No guarda tipos | Sí | Sí, y lo valida |
| Lectura columnar | No | Sí | Sí |
| ACID | No | No | **Sí** |
| Time travel | No | No | **Sí** |

## ✔ Checkpoint

- [ ] Medí que Parquet/Delta pesan menos que CSV
- [ ] Vi `_delta_log` y sé que Delta = Parquet + log
- [ ] Comprobé que una escritura fallida no altera la tabla
- [ ] Recuperé datos con `versionAsOf` y `RESTORE`

**Siguiente:** `05_Bronze_multiples_archivos`